## 🏗️ Delta Tables en Databricks

### ¿Qué convierte un simple archivo Parquet en una plataforma transaccional?

En este notebook construiremos y analizaremos Delta Tables desde cero para entender:

* ✅ Cómo se crean
* ✅ Cómo se almacenan
* ✅ Qué papel juega Unity Catalog
* ✅ Por qué Delta Lake aporta capacidades ACID

### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("01DeltaTables").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### 📚 Definición de una Delta Table

Una Delta Table es una tabla construida sobre Delta Lake que combina:
* 📦 Archivos Parquet para almacenar datos
* 📝 Transaction Logs para registrar cambios
* 🔒 Garantías ACID para proteger la consistencia

Gracias a esta arquitectura podemos disponer de:
* ✅ Versionado
* ✅ Time Travel
* ✅ Actualizaciones y eliminaciones eficientes
* ✅ Mayor confiabilidad para pipelines de datos

Asimismo, debemos tener en cuenta la siguiente información:
[Objetos_Databricks.md](https://github.com/BrayanR03/Databricks-DE-2026/tree/main/assets/definicion_objetos_databricks.md)

#### 🛡️ Gobernanza con Unity Catalog

Antes de crear una Delta Table debemos definir dónde vivirá dentro de la estructura de gobernanza, en `Unity Catalog`, el cuál organiza los datos mediante tres niveles:

* 📂 Catálogo (`Catalog`)
* 📂 Esquema (`Schema`)
* 📂 Tabla o Volumen (`Table o Volume`)

* Estructura general:
    * `catalog`.`schema`.`table`

Para nuestro ejemplo de hoy, crearemos:
- `catalog_databricks_2026_de`.`schema_databricks_2026_de`
- La tabla lo definiremos en las siguientes celdas.

In [0]:
## DEFINIREMOS NUESTRA ESTRUCTURA DE UNITY CATALOG MEDIANTE SPARK.SQL
### 1. Definimos Catalago
spark.sql("CREATE CATALOG IF NOT EXISTS catalog_databricks_2026_de")
print("Catalago creado correctamente")
### 2. Definimos Esquema
spark.sql("CREATE SCHEMA IF NOT EXISTS catalog_databricks_2026_de.schema_databricks_2026_de")
print("Esquema creado corretamente")

##💡La tabla delta se define posteriormente a la definición de la estructura inicial de Unity Catalog 

### ⚡ Vamos a crear nuestra primera Delta Table

Hasta ahora únicamente hemos revisado la teoría.

Ahora construiremos una Delta Table paso a paso para observar qué ocurre internamente dentro de Databricks.


In [0]:
## DEFINIREMOS LA TABLA MEDIANTE SPARK.SQL
spark.sql("""
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table
          (
              customer_id INT,
              first_name VARCHAR(30),
              last_name VARCHAR(30),
              country VARCHAR(30),
              email VARCHAR(30)
          ) USING DELTA;
          """)
"""
    💡 Tener en cuenta que, al definir una delta table desde cero (como una tabla en un motor de BD relacional)
        debemos utilizar USING DELTA después de los paréntesis para que Databricks la defina como tal.

        Y si en caso, no colocamos USING DELTA, por defecto Databricks la define como tal. 
"""

print("Delta table customers creada correctamente")

In [0]:
## INSERTAREMOS DATOS EN NUESTRA DELTA TABLE DEFINIDA PREVIAMENTE MEDIANTE SPARK.SQL

spark.sql("""
          INSERT INTO catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table
          VALUES (1, 'Brayan','Neciosup','Peru','brayan@gmail.com');
          """)
print("Datos registrados correctamente")

### 🤯 ¿Qué ocurre después de crear una Delta Table?

A simple vista parece que únicamente hemos almacenado datos. Sin embargo, internamente Databricks ha construido una estructura mucho más sofisticada que una simple colección de archivos Parquet. Y, antes de explorar los archivos físicos que componen nuestra Delta Table, realizaremos una primera inspección utilizando dos comandos fundamentales:

🔍 **DESCRIBE DETAIL `nombre_catalago`.`nombre_esquema`.`nombre_tabla`**
- Permite consultar información general de la tabla.
- Muestra detalles como ubicación, formato, tamaño, número de archivos y metadatos asociados.

📜 **DESCRIBE HISTORY `nombre_catalago`.`nombre_esquema`.`nombre_tabla`**
- Permite consultar el historial transaccional de la tabla.
- Muestra las operaciones ejecutadas sobre ella, quién las realizó y cuándo ocurrieron.

Gracias a estos comandos podremos obtener una primera visión de nuestra Delta Table sin necesidad de acceder directamente a su estructura interna.

Posteriormente profundizaremos en:
* 📦 Cómo se almacenan físicamente los datos
* 📝 Cómo Delta Lake registra cada transacción mediante el _delta_log
* 🔄 Cómo se construye el historial de versiones

⏳ Y cómo toda esta arquitectura hace posible una de las funcionalidades más potentes de Delta Lake: **Time Travel**

Veamos qué se ha creado detrás de escena.

In [0]:
## DESCRIBE DETAIL
display(spark.sql("DESCRIBE DETAIL catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))
"""
    Los campos mas importantes de DESCRIBE DETAIL son:
    . format: Formato que almacena los datos la tabla
    . id: ID de la tabla
    . name: estructura completa que gobierna la tabla
    . createdAt: Fecha de creación
    . lastModified: Fecha de modificación
"""
## 💡 display() permite mostrar diferentes campos en un formato amigable (tabla).
print()

In [0]:
## DESCRIBE HISTORY
display(spark.sql("DESCRIBE HISTORY catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table"))
"""
    Los campos mas importantes de DESCRIBE HISTORY son:
    . version: Version de la tabla desde su definición (varia según las operaciones aplicadas)
    . timestamp: Fecha y hora de creación de la versión de la tabla
    . userid: ID del usuario que generó versión de la delta table
    . operation: Operación realizada por cada versión de la tabla.
    . operationParameters: Parametros adicionales agregados en la definición y/o operación de la delta table
"""
## 💡 display() permite mostrar diferentes campos en un formato amigable (tabla).
print()